<a href="https://colab.research.google.com/github/IshiPareek/mech_interpet/blob/main/01_gpt2_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — imports
!pip install transformer_lens
import transformer_lens
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
import torch
import numpy as np
from jaxtyping import Float, Int
import einops

print(f"TransformerLens installd")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
model = HookedTransformer.from_pretrained("gpt2")

In [ ]:
tokens = model.to_tokens("The meaning of life is")
str_tokens = model.to_str_tokens(tokens)
print(model.to_str_tokens(tokens))
logits = model(tokens)
print(logits.shape)

In [ ]:
# ── VISUALIZATION BOILERPLATE ── define once, call anywhere ──────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def viz(cache, plot_type, layer=0, head=0, dims=64, model=model, str_tokens=str_tokens, original_loss=None, ablated_loss=None):
    """
    Universal cache visualizer.

    plot_type options:
        "embed"      → token embeddings
        "pos_embed"  → positional embeddings
        "pattern"    → attention pattern for one head
        "all_heads"  → all 12 heads in a layer
        "resid"      → residual stream at a layer
        "logit_lens" → top predicted token across all layers
        "qkv"        → query, key, value vectors
    """

    def heatmap(ax, matrix, title, row_labels=None, cmap="RdBu_r"):
        vmax = np.abs(matrix).max()
        ax.imshow(matrix, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(title, fontsize=10, pad=6)
        if row_labels:
            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels, fontsize=8)
        else:
            ax.set_yticks([])
        ax.set_xticks([])

    # ── embed ────────────────────────────────────────────────────────────────
    if plot_type == "embed":
        data = cache["hook_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_embed — first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── pos_embed ────────────────────────────────────────────────────────────
    elif plot_type == "pos_embed":
        data = cache["hook_pos_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_pos_embed — first {dims} dims", row_labels=str_tokens, cmap="PiYG")
        plt.tight_layout(); plt.show()

    # ── pattern ──────────────────────────────────────────────────────────────
    elif plot_type == "pattern":
        data = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().numpy()
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(str_tokens))); ax.set_xticklabels(str_tokens, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(str_tokens))); ax.set_yticklabels(str_tokens, fontsize=8)
        ax.set_title(f"attention pattern — layer {layer}, head {head}", fontsize=10)
        plt.tight_layout(); plt.show()

    # ── all_heads ────────────────────────────────────────────────────────────
    elif plot_type == "all_heads":
        fig, axes = plt.subplots(3, 4, figsize=(14, 9))
        for h, ax in enumerate(axes.flat):
            data = cache[f"blocks.{layer}.attn.hook_pattern"][0, h].detach().numpy()
            ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
            ax.set_title(f"H{h}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(f"all attention heads — layer {layer}", fontsize=12)
        plt.tight_layout(); plt.show()

    # ── resid ────────────────────────────────────────────────────────────────
    elif plot_type == "resid":
        data = cache[f"blocks.{layer}.hook_resid_post"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_resid_post — layer {layer}, first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── logit_lens ───────────────────────────────────────────────────────────
    elif plot_type == "logit_lens":
        n_layers = model.cfg.n_layers
        cell_text = []
        for l in range(n_layers):
            resid = cache[f"blocks.{l}.hook_resid_post"]
            top = model.unembed(model.ln_final(resid))[0, :, :].argmax(dim=-1)
            cell_text.append([model.to_string(t) for t in top])
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.axis("off")
        tbl = ax.table(cellText=cell_text, rowLabels=[f"L{l}" for l in range(n_layers)],
                       colLabels=str_tokens, loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
        ax.set_title("logit lens — top predicted token per layer × position", fontsize=11, pad=20)
        plt.tight_layout(); plt.show()

    # ── qkv ──────────────────────────────────────────────────────────────────
    elif plot_type == "qkv":
        fig, axes = plt.subplots(1, 3, figsize=(14, 3))
        for ax, key, label in zip(axes, ["hook_q","hook_k","hook_v"], ["Query","Key","Value"]):
            data = cache[f"blocks.{layer}.attn.{key}"][0, :, head, :].detach().numpy()
            heatmap(ax, data, f"{label} — layer {layer}, head {head}", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

## ABLATION
    elif plot_type == "ablation":
      if original_loss is None or ablated_loss is None :
        print ("pass original_loss and ablation_loss to use this plot type")
        return

      orig = original_loss.item()
      abla = ablated_loss.item()
      diff = abla - orig

      fig, ax = plt.subplots(figsize=(5, 4))
      bars = ax.bar(
      ["Original Loss", "Ablated Loss"],
      [orig, abla],
      color=["steelblue", "tomato"])

      ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
      ax.set_title(
            f"Head Ablation — Layer {layer}, Head {head}\nΔ Loss = +{diff:.3f}",
      fontsize=11)
      ax.set_ylabel("Cross Entropy Loss")
      ax.set_ylim(0, max(orig, abla) * 1.2)
      plt.tight_layout(); plt.show()

    else:
        print(f"unknown plot_type '{plot_type}'. options: embed, pos_embed, pattern, all_heads, resid, logit_lens, qkv")

In [ ]:
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
print(list(cache.keys())[:10])

In [ ]:
resid_mid = cache["blocks.5.hook_resid_post"]
print(resid_mid.shape)

In [ ]:
cache ['hook_embed']
#print(model.to_str_tokens(tokens))

In [ ]:
cache ["hook_pos_embed"]

In [ ]:
cache ['blocks.1.hook_resid_pre']


In [ ]:
cache ['blocks.2.hook_resid_pre']

In [ ]:
logits, cache = model.run_with_cache(tokens)
cache ['blocks.2.hook_resid_pre']

In [ ]:
attention_pattern = cache["pattern", 0, "attn"]
print(attention_pattern.shape)
print(model.to_str_tokens(tokens))

In [ ]:
head_0 = cache["attn", 0, "pattern"]
print (head_0)
print(model.to_str_tokens(tokens))

In [ ]:
head_1 = cache["attn", 4, "pattern"]
print (head_1)
print(model.to_str_tokens(tokens))
viz (cache, "pattern", layer=4, head=0)

In [ ]:
str_tokens = model.to_str_tokens(tokens)
viz(cache, "pattern", layer=4, head=11)  # previous token head
viz(cache, "pattern", layer=4, head=8)   # attention sink head

In [ ]:
prompts = [
    "The meaning of life is",
    "The cat sat on the mat",
    "The president of the United States",
]

for prompt in prompts:
    tokens = model.to_tokens(prompt)
    logits, cache = model.run_with_cache(tokens)
    pattern = cache["blocks.0.attn.hook_pattern"][0]  # [12, seq, seq]
    str_tokens = model.to_str_tokens(tokens)

    print(f"\nPrompt: '{prompt}'")
    print(f"Tokens: {str_tokens}")
    print(f"Head 0 — last token attention: {pattern[0, -1].detach().numpy().round(3)}")
    print(f"Head 1 — last token attention: {pattern[1, -1].detach().numpy().round(3)}")